# AgriWatch Monitor — ciągły monitoring wilgotności gleby (Condom, Gers)
**Jak używać:** raz w tygodniu `Środowisko wykonawcze → Uruchom wszystko`. Nie trzeba niczego zmieniać w komórkach.

- Kod: GitHub → czysty klon na Dysku `MyDrive/GeoWorldLook/agriwatch` (`git pull` w komórce 1). Otwieraj ten notatnik z GitHub, nie z Dysku.
- Obliczenia: ten notatnik w Colab (wystarczy CPU).
- Wyniki: tabele `data/registry/gwl_*.csv` na Dysku (schemat gotowy do eksportu do bazy danych) oraz raporty w `data/08_Station_Validation/`.
- Każde zadanie jest zapisane w `gwl_runs` (czas, commit, liczba nowych wierszy, błąd). Błąd jednego zadania nie zatrzymuje kolejnych.

Plan: `docs/plans/Plan_v3_monitoring_winnic_SR.md`. Etap E0: stacja Condom.


## 1. Dysk Google, kod z GitHub


In [ ]:
import os, sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

REPO_URL = 'https://github.com/geoworldlook/1_geoworldlook-agriscreen.git'
# Jedyny folder projektu na Dysku: czysty klon repozytorium. Notatnik otwieraj z GitHub (Plik -> Otwórz -> GitHub),
# żeby Colab nie zapisywał zmian w tym folderze (inaczej git pull odmówi aktualizacji).
PROJECT_DIR = '/content/drive/MyDrive/GeoWorldLook/agriwatch'
if IN_COLAB:
    if not os.path.exists(os.path.join(PROJECT_DIR, '.git')):
        os.makedirs(os.path.dirname(PROJECT_DIR), exist_ok=True)
        !git clone {REPO_URL} "{PROJECT_DIR}"
    !git -C "{PROJECT_DIR}" pull --ff-only
    if not os.path.exists(os.path.join(PROJECT_DIR, 'step_07_station_pipeline.py')):
        raise RuntimeError('Brak step_07 po git pull — sprawdź komunikat git powyżej.')
else:
    # Lokalnie: notatnik leży w notebooks/, projekt piętro wyżej
    PROJECT_DIR = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.path.abspath('.')

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
# autoreload (przeładowanie kodu po git pull bez restartu); Python 3.12+ nie ma modułu 'imp'
import importlib
sys.modules.setdefault('imp', importlib)
try:
    %load_ext autoreload
    %autoreload 2
except Exception as e:
    print(f'[INFO] autoreload pominięty ({e}) — po git pull zrestartuj sesję.')
print('Projekt:', PROJECT_DIR)


## 2. Środowisko: pakiety, Google Earth Engine, rejestr


In [ ]:
from step_05_colab_run import setup_runtime, run_task, run_history, registry_summary, registry_read

rt = setup_runtime(PROJECT_DIR, gee_project='ee-geoworldlook', pull=False)


## 3. Stacja Condom — wilgotność z Sentinel-1 w miejscu czujnika i błąd względem pomiaru 5 cm
Dane S-1, S-2 i ERA5-Land są pobierane z GEE z cache na Dysku (kolejne uruchomienia pobierają tylko brakujące okresy).
Wynik: raport, wykres, metryki z 95% CI oraz wiersze w `gwl_observations`, `gwl_calibrations`, `gwl_validation_metrics`.


In [ ]:
from step_07_station_pipeline import run_station_pipeline

station = run_task(rt, 'station_condom', run_station_pipeline, rt)


## 4. Panel: wyniki, rejestr, historia uruchomień


In [ ]:
from IPython.display import display, Image, Markdown

if station:
    display(Image(filename=station['plot_path']))
    with open(station['report_path'], 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))

display(registry_summary(rt))
display(run_history(rt, n=10))
